# Normalizaci?n y EDA de impagos

Este notebook usa solo dos archivos de entrada:

- Variable Y.csv
- EEFF base de datos.csv

A partir de ellos calcula la columna Y, la fusiona con la tabla EEFF por ID y crea el dataframe fusionado para el resto del an?lisis.


In [ ]:
# Ejecuta esta celda si tu entorno no tiene las librerías instaladas.
%pip install pandas numpy matplotlib seaborn

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)


def normalizar_id(serie):
    serie_texto = serie.astype('string').str.strip()
    serie_num = pd.to_numeric(serie_texto, errors='coerce')
    return serie_num.astype('Int64').astype('string').where(serie_num.notna(), serie_texto)


ruta_variable_y = Path('Variable Y.csv')

variable_y_raw = pd.read_csv(ruta_variable_y, sep=';', encoding='utf-8-sig', low_memory=False)

if 'ID' not in variable_y_raw.columns:
    raise KeyError('No se encontr? la columna ID en Variable Y.csv')

# Si el CSV ya trae una Y de una ejecuci?n anterior, se recalcula desde las columnas originales.
columnas_para_y = [col for col in variable_y_raw.columns if col not in {'ID', 'Y'}]
variable_y_raw['Y'] = (
    variable_y_raw[columnas_para_y]
    .astype('string')
    .apply(lambda fila: fila.str.strip().str.upper().eq('VERDADERO').any(), axis=1)
    .astype(int)
)
variable_y_raw['ID'] = normalizar_id(variable_y_raw['ID'])

# Se eliminan columnas sin nombre, vac?as o completamente nulas solo despu?s de haber calculado Y.
variable_y = variable_y_raw.loc[:, ~variable_y_raw.columns.astype(str).str.match(r'^Unnamed')].copy()
variable_y = variable_y.loc[:, variable_y.columns.astype(str).str.strip() != '']
variable_y = variable_y.dropna(axis=1, how='all')
variable_y = variable_y.dropna(axis=0, how='all').copy()

# Mantiene df como alias para las celdas iniciales de exploraci?n de Variable Y.
df = variable_y.copy()

print(f'Variable Y -> filas: {variable_y.shape[0]}, columnas: {variable_y.shape[1]}')
print(f'Y=1: {int((variable_y["Y"] == 1).sum())}')
print(f'% Y=1: {variable_y["Y"].mean() * 100:.4f}%')
variable_y.head()


In [ ]:
# La variable objetivo Y se calcula desde Variable Y.csv.
# No se carga ning?n CSV fusionado externo.
resumen_y_variable = pd.Series({
    'filas': len(variable_y),
    'Y=1': int((variable_y['Y'] == 1).sum()),
    'Y=0': int((variable_y['Y'] == 0).sum()),
    '% Y=1': variable_y['Y'].mean() * 100
})
resumen_y_variable


In [ ]:
df_normalizado = variable_y.copy()

columnas_periodo = [col for col in df_normalizado.columns if col not in {'ID', 'Y'}]

mapeo_binario = {
    'VERDADERO': 1,
    'FALSO': 0,
    '#?NULO!': np.nan,
    '': np.nan
}

df_normalizado[columnas_periodo] = df_normalizado[columnas_periodo].replace(mapeo_binario)
df_normalizado['ID'] = pd.to_numeric(df_normalizado['ID'], errors='coerce')
df_normalizado['Y'] = pd.to_numeric(df_normalizado['Y'], errors='coerce')

# Convierte las columnas mensuales a tipo num?rico para asegurar 0/1.
df_normalizado[columnas_periodo] = df_normalizado[columnas_periodo].apply(pd.to_numeric, errors='coerce')

df_normalizado.head()


In [ ]:
# No se exportan CSV intermedios: el notebook crea los dataframes en memoria
# a partir de Variable Y.csv y EEFF base de datos.csv.
variable_y[['ID', 'Y']].head()


In [ ]:
meses_es = {
    'Enero': '01',
    'Febrero': '02',
    'Marzo': '03',
    'Abril': '04',
    'Mayo': '05',
    'Junio': '06',
    'Julio': '07',
    'Agosto': '08',
    'Septiembre': '09',
    'Octubre': '10',
    'Noviembre': '11',
    'Diciembre': '12'
}

df_largo = df_normalizado.melt(
    id_vars='ID',
    value_vars=columnas_periodo,
    var_name='periodo',
    value_name='impago'
)

periodo_split = df_largo['periodo'].str.extract(r'(?P<mes>\w+)\s+(?P<anio>\d{4})')
df_largo['mes'] = periodo_split['mes']
df_largo['anio'] = pd.to_numeric(periodo_split['anio'], errors='coerce')
df_largo['fecha'] = pd.to_datetime(
    periodo_split['anio'].astype(str) + '-' + periodo_split['mes'].map(meses_es) + '-01',
    errors='coerce'
)

porcentaje_total = df_largo['impago'].mean() * 100

porcentaje_por_mes_anio = (
    df_largo.dropna(subset=['fecha'])
    .groupby('fecha', as_index=False)['impago']
    .mean()
    .assign(porcentaje_impago=lambda x: x['impago'] * 100)
    .sort_values('fecha')
)

porcentaje_por_anio = (
    df_largo.dropna(subset=['anio'])
    .groupby('anio', as_index=False)['impago']
    .mean()
    .assign(porcentaje_impago=lambda x: x['impago'] * 100)
    .sort_values('anio')
)

print(f'Porcentaje total de impagos: {porcentaje_total:.2f}%')

display(
    porcentaje_por_mes_anio[['fecha', 'porcentaje_impago']]
    .rename(columns={'fecha': 'mes_anio', 'porcentaje_impago': 'pct_impago'})
)

display(
    porcentaje_por_anio[['anio', 'porcentaje_impago']]
    .rename(columns={'porcentaje_impago': 'pct_impago'})
)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

sns.barplot(
    data=porcentaje_por_mes_anio,
    x='fecha',
    y='porcentaje_impago',
    color='#D55E00',
    ax=axes[0]
)
axes[0].set_title('Porcentaje de impagos por mes/año')
axes[0].set_xlabel('Mes/Año')
axes[0].set_ylabel('% de impago')
axes[0].tick_params(axis='x', rotation=45)

sns.barplot(
    data=porcentaje_por_anio,
    x='anio',
    y='porcentaje_impago',
    color='#0072B2',
    ax=axes[1]
)
axes[1].set_title('Porcentaje de impagos por año')
axes[1].set_xlabel('Año')
axes[1].set_ylabel('% de impago')

plt.tight_layout()
plt.show()

## EDA de la tabla EEFF

Esta sección carga la tabla `EEFF` desde `CSV`, sustituye marcadores de ausentes como `n.d.` y `N.D.` por valor nulo y resume los tipos de variables para distinguir identificadores, variables categóricas y variables numéricas.


In [ ]:
ruta_eeff = Path('EEFF base de datos.csv')

df_eeff_original = pd.read_csv(ruta_eeff, sep=';', encoding='utf-8-sig')
df_eeff_original = df_eeff_original.loc[:, ~df_eeff_original.columns.astype(str).str.match(r'^Unnamed')]
df_eeff_original = df_eeff_original.loc[:, df_eeff_original.columns.astype(str).str.strip() != '']

id_eeff = 'Id' if 'Id' in df_eeff_original.columns else 'ID'
if id_eeff not in df_eeff_original.columns:
    raise KeyError('No se encontr? la columna Id/ID en EEFF base de datos.csv')

# Si EEFF base de datos.csv ya trae Y de una ejecuci?n previa, se elimina para rehacer la fusi?n desde cero.
df_eeff_sin_y = df_eeff_original.drop(columns=['Y'], errors='ignore').copy()
df_eeff_sin_y = df_eeff_sin_y.rename(columns={id_eeff: 'ID'})

variable_y_para_merge = variable_y[['ID', 'Y']].copy()
variable_y_para_merge['ID'] = normalizar_id(variable_y_para_merge['ID'])
df_eeff_sin_y['ID'] = normalizar_id(df_eeff_sin_y['ID'])

# Dataframe fusionado: todas las columnas de EEFF + solo la columna Y calculada desde Variable Y.csv.
df_fusionado = df_eeff_sin_y.merge(variable_y_para_merge, on='ID', how='left')
df_fusionado['Y'] = pd.to_numeric(df_fusionado['Y'], errors='coerce')

# Alias usados por el resto del notebook.
eeff_con_variable_y = df_fusionado.copy()
df_eeff = df_fusionado.copy()

with pd.option_context('future.no_silent_downcasting', True):
    df_eeff = df_eeff.replace({'n.d.': np.nan, 'N.D.': np.nan, '': np.nan})
df_eeff = df_eeff.infer_objects(copy=False)

def parsear_numero_eeff(valor):
    if pd.isna(valor):
        return np.nan

    texto = str(valor).strip()
    if texto == '':
        return np.nan

    if '.' in texto and ',' in texto:
        texto = texto.replace('.', '').replace(',', '.')
    elif ',' in texto:
        texto = texto.replace(',', '.')
    elif '.' in texto:
        if pd.Series([texto]).str.match(r'^-?\d{1,3}(\.\d{3})+$').iloc[0]:
            texto = texto.replace('.', '')

    try:
        return float(texto)
    except ValueError:
        return np.nan

print(f'df_fusionado -> filas: {df_fusionado.shape[0]}, columnas: {df_fusionado.shape[1]}')
print(f'IDs sin Y: {int(df_fusionado["Y"].isna().sum())}')
print(f'Y=1: {int((df_fusionado["Y"] == 1).sum())}')
print(f'% Y=1: {df_fusionado["Y"].mean() * 100:.4f}%')
df_fusionado.head()


In [ ]:
conteo_ids = eeff_con_variable_y['ID'].value_counts(dropna=False)
ids_con_4_registros = conteo_ids[conteo_ids == 4].index

df_4 = eeff_con_variable_y[eeff_con_variable_y['ID'].isin(ids_con_4_registros)].copy()
df_4['Y'] = pd.to_numeric(df_4['Y'], errors='coerce')

print(f'df_4 -> filas: {df_4.shape[0]}, IDs ?nicos: {df_4["ID"].nunique()}')
print(f'Y=1: {int((df_4["Y"] == 1).sum())}')
print(f'% Y=1: {df_4["Y"].mean() * 100:.4f}%')
df_4.head()


In [ ]:
df_eeff_tipado = df_eeff.copy()

for col in df_eeff_tipado.columns:
    if df_eeff_tipado[col].dtype == 'object':
        serie_original = df_eeff_tipado[col]
        serie_parseada = serie_original.map(parsear_numero_eeff)
        no_nulos_original = serie_original.notna().sum()
        ratio_conversion = 0 if no_nulos_original == 0 else serie_parseada.notna().sum() / no_nulos_original

        if ratio_conversion >= 0.95:
            df_eeff_tipado[col] = serie_parseada

def clasificar_variable(serie, nombre_columna):
    serie_no_nula = serie.dropna()
    nombre = str(nombre_columna).strip().lower()

    if nombre in {'id', 'identifier'} or nombre.startswith('id'):
        return 'Identificador'

    if pd.api.types.is_numeric_dtype(serie):
        if serie_no_nula.empty:
            return 'Numérica'

        valores_unicos = serie_no_nula.nunique()
        es_entera = np.all(np.isclose(serie_no_nula % 1, 0))

        if valores_unicos == 2:
            return 'Binaria numérica'
        if es_entera and valores_unicos <= 20:
            return 'Numérica discreta'
        if es_entera:
            return 'Numérica entera'
        return 'Numérica continua'

    valores_unicos = serie_no_nula.nunique()
    if valores_unicos == 2:
        return 'Binaria categórica'
    return 'Categórica'

resumen_variables = pd.DataFrame({
    'variable': df_eeff_tipado.columns,
    'dtype_pandas': [str(df_eeff_tipado[col].dtype) for col in df_eeff_tipado.columns],
    'tipo_variable': [clasificar_variable(df_eeff_tipado[col], col) for col in df_eeff_tipado.columns],
    'nulos_pct': [df_eeff_tipado[col].isna().mean() * 100 for col in df_eeff_tipado.columns],
    'valores_unicos': [df_eeff_tipado[col].nunique(dropna=True) for col in df_eeff_tipado.columns],
    'ejemplo_valores': [', '.join(map(str, df_eeff_tipado[col].dropna().astype(str).unique()[:3])) for col in df_eeff_tipado.columns]
}).sort_values(['tipo_variable', 'nulos_pct', 'valores_unicos'], ascending=[True, False, False])

display(resumen_variables)

display(
    resumen_variables.groupby('tipo_variable', as_index=False)
    .agg(numero_variables=('variable', 'count'))
    .sort_values('numero_variables', ascending=False)
)


In [ ]:
conteo_tipos = (
    resumen_variables['tipo_variable']
    .value_counts()
    .rename_axis('tipo_variable')
    .reset_index(name='numero_variables')
)

plt.figure(figsize=(10, 5))
sns.barplot(data=conteo_tipos, x='tipo_variable', y='numero_variables', palette='Blues_r')
plt.title('Distribuci?n de tipos de variables en EEFF')
plt.xlabel('Tipo de variable')
plt.ylabel('N?mero de variables')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

resumen_variables.groupby('tipo_variable')['nulos_pct'].describe().round(2)


In [ ]:
# COMPROBAR QUE SALEN 4 IDs

df_4['ID'].value_counts().eq(4).all()


## trata de nulos

Este bloque aplica sobre `df_4` la estrategia de `tratamiento_valores_nulos.md`: tipa columnas numéricas, reconstruye totales contables, crea indicadores binarios y completa el resto de partidas financieras.

In [ ]:
df_4_tratado = df_4.copy()

with pd.option_context('future.no_silent_downcasting', True):
    df_4_tratado = df_4_tratado.replace({'n.d.': np.nan, 'N.D.': np.nan, '': np.nan})

for col in df_4_tratado.columns:
    if df_4_tratado[col].dtype == 'object':
        serie_original = df_4_tratado[col]
        serie_parseada = serie_original.map(parsear_numero_eeff)
        no_nulos_original = serie_original.notna().sum()
        ratio_conversion = 0 if no_nulos_original == 0 else serie_parseada.notna().sum() / no_nulos_original

        if ratio_conversion >= 0.95:
            df_4_tratado[col] = serie_parseada

def normalizar_nombre_columna(nombre):
    return ' '.join(str(nombre).replace('\r', ' ').replace('\n', ' ').split()).lower()

def buscar_columna(df_base, fragmento, preferir_ultima=False):
    fragmento_normalizado = normalizar_nombre_columna(fragmento)
    candidatas = [
        col for col in df_base.columns
        if fragmento_normalizado in normalizar_nombre_columna(col)
    ]

    if not candidatas:
        raise KeyError(f'No se encontró una columna que contenga: {fragmento}')

    return candidatas[-1] if preferir_ultima else candidatas[0]

col_id = buscar_columna(df_4_tratado, 'ID')
col_anio = buscar_columna(df_4_tratado, 'Año')
col_cnae = buscar_columna(df_4_tratado, 'Código primario CNAE 2009')
col_num_empleados = buscar_columna(df_4_tratado, 'Número empleados')

col_activo_no_corriente = buscar_columna(df_4_tratado, 'Activo no corriente')
col_activo_corriente = buscar_columna(df_4_tratado, 'Activo corriente')
col_total_activo = buscar_columna(df_4_tratado, 'Total activo (A + B)')
col_patrimonio_neto = buscar_columna(df_4_tratado, 'Patrimonio neto')
col_pasivo_no_corriente = buscar_columna(df_4_tratado, 'Pasivo no corriente')
col_pasivo_corriente = buscar_columna(df_4_tratado, 'Pasivo corriente')
col_total_pn_pasivo = buscar_columna(df_4_tratado, 'Total patrimonio neto y pasivo (A + B + C)')

df_4_tratado[col_total_activo] = df_4_tratado[col_total_activo].fillna(
    df_4_tratado[col_activo_no_corriente] + df_4_tratado[col_activo_corriente]
)
df_4_tratado[col_total_pn_pasivo] = df_4_tratado[col_total_pn_pasivo].fillna(
    df_4_tratado[col_patrimonio_neto] + df_4_tratado[col_pasivo_no_corriente] + df_4_tratado[col_pasivo_corriente]
)

col_resultado_ejercicio = buscar_columna(df_4_tratado, 'Resultado del ejercicio')
col_gastos_personal = buscar_columna(df_4_tratado, 'Gastos de personal')
col_amortizacion = buscar_columna(df_4_tratado, 'Amortización del inmovilizado')
col_cifra_negocios = buscar_columna(df_4_tratado, 'Importe neto de la cifra de negocios')
col_resultado_explotacion = buscar_columna(df_4_tratado, 'Resultado de explotación (1 + 2 + 3 + 4 + 5 + 6 + 7 + 8 + 9 + 10 + 11 + 12)')
col_resultado_antes_impuestos = buscar_columna(df_4_tratado, 'Resultado antes de impuestos (A + B)')

cols_clave = [
    col_activo_corriente,
    col_activo_no_corriente,
    col_resultado_ejercicio,
    col_gastos_personal,
    col_amortizacion,
    col_cifra_negocios,
    col_resultado_explotacion,
    col_resultado_antes_impuestos
]

for col in cols_clave:
    mediana_id_anio = df_4_tratado.groupby([col_id, col_anio])[col].transform('median')
    mediana_anio_cnae = df_4_tratado.groupby([col_anio, col_cnae])[col].transform('median')
    mediana_anio = df_4_tratado.groupby(col_anio)[col].transform('median')
    df_4_tratado[col] = (
        df_4_tratado[col]
        .fillna(mediana_id_anio)
        .fillna(mediana_anio_cnae)
        .fillna(mediana_anio)
    )

col_existencias = buscar_columna(df_4_tratado, 'Existencias')
col_tesoreria = buscar_columna(df_4_tratado, 'Efectivo y otros activos líquidos equivalentes')
col_subvenciones = buscar_columna(df_4_tratado, 'Subvenciones, donaciones y legados recibidos')
col_inv_grupo_lp = buscar_columna(df_4_tratado, 'Inversiones en empresas del grupo y asociadas a largo plazo')
col_arrendamiento_financiero = buscar_columna(df_4_tratado, 'Acreedores por arrendamiento financiero')
col_deudas_largo_plazo = buscar_columna(df_4_tratado, 'Deudas a largo plazo')
col_deudas_corto_plazo = buscar_columna(df_4_tratado, 'Deudas a corto plazo')
col_otros_ingresos_explotacion = buscar_columna(df_4_tratado, 'Otros ingresos de explotación')
col_ingresos_financieros = buscar_columna(df_4_tratado, 'Ingresos financieros')
col_aprovisionamientos = buscar_columna(df_4_tratado, 'Aprovisionamientos')
col_otros_gastos_explotacion = buscar_columna(df_4_tratado, 'Otros gastos de explotación')

pasivo_corriente_valido = df_4_tratado[col_pasivo_corriente].replace(0, np.nan)
df_4_tratado['Current Ratio'] = df_4_tratado[col_activo_corriente] / pasivo_corriente_valido
df_4_tratado['Quick Ratio'] = (df_4_tratado[col_activo_corriente] - df_4_tratado[col_existencias]) / pasivo_corriente_valido
df_4_tratado['Cash Ratio'] = df_4_tratado[col_tesoreria] / pasivo_corriente_valido

deuda_total_proxy = df_4_tratado[col_pasivo_no_corriente].fillna(0) + df_4_tratado[col_pasivo_corriente].fillna(0)
deuda_financiera_proxy = df_4_tratado[col_deudas_largo_plazo].fillna(0) + df_4_tratado[col_deudas_corto_plazo].fillna(0) + df_4_tratado[col_arrendamiento_financiero].fillna(0)
ingresos_brutos_anuales_proxy = (df_4_tratado[col_cifra_negocios].fillna(0) + df_4_tratado[col_otros_ingresos_explotacion].fillna(0) + df_4_tratado[col_ingresos_financieros].fillna(0)).replace(0, np.nan)
activos_totales_validos = df_4_tratado[col_total_activo].replace(0, np.nan)
ebitda_proxy = (df_4_tratado[col_resultado_explotacion].fillna(0) + df_4_tratado[col_amortizacion].fillna(0)).replace(0, np.nan)
patrimonio_neto_valido = df_4_tratado[col_patrimonio_neto].replace(0, np.nan)
gastos_operativos_anuales_proxy = (df_4_tratado[col_aprovisionamientos].abs().fillna(0) + df_4_tratado[col_gastos_personal].abs().fillna(0) + df_4_tratado[col_otros_gastos_explotacion].abs().fillna(0)).replace(0, np.nan)
gastos_operativos_diarios_proxy = gastos_operativos_anuales_proxy / 365

df_4_tratado['DTI total'] = deuda_total_proxy / ingresos_brutos_anuales_proxy
df_4_tratado['Deuda total / Activos totales'] = deuda_total_proxy / activos_totales_validos
df_4_tratado['Deuda neta / EBITDA'] = (deuda_financiera_proxy - df_4_tratado[col_tesoreria].fillna(0)) / ebitda_proxy
df_4_tratado['Deuda financiera / Patrimonio neto'] = deuda_financiera_proxy / patrimonio_neto_valido
df_4_tratado['Days Cash on Hand'] = df_4_tratado[col_tesoreria] / gastos_operativos_diarios_proxy

df_4_tratado['recibe_subvenciones'] = df_4_tratado[col_subvenciones].notna().astype(int)
df_4_tratado['tiene_inversiones_grupo'] = df_4_tratado[col_inv_grupo_lp].notna().astype(int)
df_4_tratado['tiene_arrendamiento_financiero'] = df_4_tratado[col_arrendamiento_financiero].notna().astype(int)

tramos_empleados = pd.cut(
    df_4_tratado[col_num_empleados],
    bins=[-np.inf, 10, 50, 250, np.inf],
    labels=['micro', 'pequeña', 'mediana', 'grande']
)
df_4_tratado[col_gastos_personal] = df_4_tratado[col_gastos_personal].fillna(
    df_4_tratado.groupby([col_anio, tramos_empleados], observed=False)[col_gastos_personal].transform('median')
)

sector_servicios = df_4_tratado[col_cnae].astype(str).str[:2].isin({'55', '56', '58', '59', '60', '61', '62', '63', '64', '66', '68', '69', '70', '71', '72', '73', '74', '77', '78', '79', '80', '81', '82', '85', '86', '87', '88', '90', '91', '92', '93', '94', '95', '96'})
mediana_existencias_industrial = df_4_tratado.groupby([col_anio, col_cnae])[col_existencias].transform('median')
df_4_tratado.loc[sector_servicios, col_existencias] = df_4_tratado.loc[sector_servicios, col_existencias].fillna(0)
df_4_tratado.loc[~sector_servicios, col_existencias] = df_4_tratado.loc[~sector_servicios, col_existencias].fillna(
    mediana_existencias_industrial[~sector_servicios]
)

cols_financieras = [col for col in df_4_tratado.columns if 'mil eur' in normalizar_nombre_columna(col)]
df_4_tratado[cols_financieras] = df_4_tratado[cols_financieras].fillna(0)

resumen_nulos_df_4 = (
    df_4_tratado.isna().mean().mul(100).round(2)
    .rename('nulos_pct')
    .reset_index()
    .rename(columns={'index': 'variable'})
    .sort_values(['nulos_pct', 'variable'], ascending=[False, True])
)

print(f'Nulos restantes en df_4_tratado: {int(df_4_tratado.isna().sum().sum())}')
display(resumen_nulos_df_4.head(15))
df_4_tratado.head()


In [ ]:
variable_año = buscar_columna(df_4_tratado, 'Año')

df_4_fusionado = (
    df_4_tratado
    .rename(columns={variable_año: 'variable_año'})
    .sort_values(['ID', 'variable_año'])
    .set_index(['ID', 'variable_año'])
    .unstack('variable_año')
)

df_4_fusionado.columns = [f'{col}_{anio}' for col, anio in df_4_fusionado.columns]
df_4_fusionado = df_4_fusionado.reset_index()

columnas_y = ['Y_2008', 'Y_2009', 'Y_2010', 'Y_2011']
if set(columnas_y).issubset(df_4_fusionado.columns):
    if not df_4_fusionado[columnas_y].nunique(axis=1, dropna=False).eq(1).all():
        raise ValueError('Las columnas Y por año no son idénticas y no se pueden fusionar en una sola Y.')
    df_4_fusionado['Y'] = df_4_fusionado['Y_2008']
    df_4_fusionado = df_4_fusionado.drop(columns=columnas_y)

ratios_base = [
    'Current Ratio',
    'Quick Ratio',
    'Cash Ratio',
    'DTI total',
    'Deuda total / Activos totales',
    'Deuda neta / EBITDA',
    'Deuda financiera / Patrimonio neto',
    'Days Cash on Hand'
]
anios_ratios = [2008, 2009, 2010, 2011]

for ratio in ratios_base:
    columnas_ratio = [f'{ratio}_{anio}' for anio in anios_ratios if f'{ratio}_{anio}' in df_4_fusionado.columns]
    if not columnas_ratio:
        continue

    col_mediana = f'{ratio}_mediana_4_anios'
    col_normalizada = f'{ratio}_mediana_4_anios_norm'
    df_4_fusionado[col_mediana] = df_4_fusionado[columnas_ratio].median(axis=1, skipna=True)

    media_ratio = df_4_fusionado[col_mediana].mean(skipna=True)
    std_ratio = df_4_fusionado[col_mediana].std(skipna=True, ddof=0)
    if pd.isna(std_ratio) or std_ratio == 0:
        df_4_fusionado[col_normalizada] = 0.0
    else:
        df_4_fusionado[col_normalizada] = (df_4_fusionado[col_mediana] - media_ratio) / std_ratio

print(f'df_4_fusionado -> filas: {df_4_fusionado.shape[0]}, IDs únicos: {df_4_fusionado["ID"].nunique()}')
col_days_cash_norm = 'Days Cash on Hand_mediana_4_anios_norm'
if col_days_cash_norm in df_4_fusionado.columns:
    df_4_fusionado['days_cash_missing'] = df_4_fusionado[col_days_cash_norm].isna().astype(int)

    columnas_cnae = [col for col in df_4_fusionado.columns if 'Código primario CNAE 2009' in col]
    if not columnas_cnae:
        raise KeyError('No se encontró ninguna columna de sector CNAE para imputar Days Cash on Hand.')

    col_sector = columnas_cnae[0]
    mediana_sector_days_cash = df_4_fusionado.groupby(col_sector)[col_days_cash_norm].transform('median')
    mediana_global_days_cash = df_4_fusionado[col_days_cash_norm].median(skipna=True)
    df_4_fusionado[col_days_cash_norm] = (
        df_4_fusionado[col_days_cash_norm]
        .fillna(mediana_sector_days_cash)
        .fillna(mediana_global_days_cash)
    )

df_4_fusionado.head()


## Feature engineering para modelo de impagos

Este bloque crea variables adicionales a partir de `df_4_fusionado`: ratios financieros, agregados temporales, tendencias, flags de deterioro, variables de auditor?a, codificaciones target out-of-fold para CNAE/provincia y variables relativas al sector. El resultado se guarda en `df_modelo_impagos` y se exporta a `fusionado_features_impagos.csv`.


In [ ]:
YEARS = [2008, 2009, 2010, 2011]
RANDOM_SEED = 42

import unicodedata
import re


def normalizar_texto_modelo(valor):
    texto = str(valor).replace('\r', ' ').replace('\n', ' ')
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('ascii')
    return re.sub(r'\s+', ' ', texto).strip().lower()


def buscar_columna_anual(df_base, fragmento, anio, ocurrencia=0):
    fragmento_normalizado = normalizar_texto_modelo(fragmento)
    sufijo = f'_{anio}'
    candidatas = [
        col for col in df_base.columns
        if col.endswith(sufijo) and fragmento_normalizado in normalizar_texto_modelo(col)
    ]
    if len(candidatas) <= ocurrencia:
        raise KeyError(f'No se encontr? columna para {fragmento!r} en {anio}. Candidatas: {candidatas[:5]}')
    return candidatas[ocurrencia]


def serie_num_anual(df_base, fragmento, anio, ocurrencia=0):
    return pd.to_numeric(df_base[buscar_columna_anual(df_base, fragmento, anio, ocurrencia)], errors='coerce')


def serie_cat_anual(df_base, fragmento, anio):
    return df_base[buscar_columna_anual(df_base, fragmento, anio)].astype('string')


def dividir_seguro(numerador, denominador):
    numerador = pd.to_numeric(numerador, errors='coerce')
    denominador = pd.to_numeric(denominador, errors='coerce')
    return numerador / denominador.where(denominador.abs() > 1e-9)


def log1p_no_negativo(serie):
    return np.log1p(pd.to_numeric(serie, errors='coerce').clip(lower=0))


def crear_folds_estratificados(y, n_splits=5, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    folds = pd.Series(index=y.index, dtype='int64')
    for _, indices_clase in y.groupby(y).groups.items():
        indices = np.array(list(indices_clase))
        rng.shuffle(indices)
        for fold, indices_fold in enumerate(np.array_split(indices, n_splits)):
            folds.loc[indices_fold] = fold
    return folds.astype(int)


def target_encoding_oof(valores, y, folds, smoothing=20.0):
    codificada = pd.Series(index=valores.index, dtype='float64')
    tasa_global = float(y.mean())
    valores = valores.astype('string').fillna('__missing__')

    for fold in sorted(folds.unique()):
        mascara_train = folds != fold
        mascara_valid = folds == fold
        estadisticos = (
            pd.DataFrame({'categoria': valores[mascara_train], 'y': y[mascara_train]})
            .groupby('categoria')['y']
            .agg(['mean', 'count'])
        )
        suavizada = (
            estadisticos['mean'] * estadisticos['count'] + tasa_global * smoothing
        ) / (estadisticos['count'] + smoothing)
        codificada.loc[mascara_valid] = valores[mascara_valid].map(suavizada).fillna(tasa_global).astype(float)

    return codificada


def racha_maxima_unos(df_flags):
    resultados = []
    for fila in df_flags.fillna(0).astype(int).to_numpy():
        actual = 0
        mejor = 0
        for valor in fila:
            actual = actual + 1 if valor == 1 else 0
            mejor = max(mejor, actual)
        resultados.append(mejor)
    return pd.Series(resultados, index=df_flags.index)


# Asegura una ?nica variable objetivo Y aunque el dataframe venga del CSV ancho original.
df_feature_base = df_4_fusionado.copy()
columnas_y_anuales = [f'Y_{anio}' for anio in YEARS]
if 'Y' not in df_feature_base.columns and set(columnas_y_anuales).issubset(df_feature_base.columns):
    if not df_feature_base[columnas_y_anuales].dropna().nunique(axis=1).le(1).all():
        raise ValueError('Las columnas Y anuales no son consistentes.')
    df_feature_base['Y'] = df_feature_base[columnas_y_anuales].bfill(axis=1).iloc[:, 0]
    df_feature_base = df_feature_base.drop(columns=columnas_y_anuales)

df_feature_base = df_feature_base[df_feature_base['Y'].notna()].copy()
df_feature_base['Y'] = df_feature_base['Y'].astype(int)

features_anuales = {}
for anio in YEARS:
    activo_corriente = serie_num_anual(df_feature_base, 'Activo corriente', anio)
    pasivo_corriente = serie_num_anual(df_feature_base, 'Pasivo corriente', anio)
    efectivo = serie_num_anual(df_feature_base, 'Efectivo y otros activos liquidos equivalentes', anio)
    existencias = serie_num_anual(df_feature_base, 'Existencias', anio)
    total_activo = serie_num_anual(df_feature_base, 'Total activo', anio)
    patrimonio_neto = serie_num_anual(df_feature_base, 'Patrimonio neto', anio)
    pasivo_no_corriente = serie_num_anual(df_feature_base, 'Pasivo no corriente', anio)
    pasivo_total = pasivo_no_corriente + pasivo_corriente
    capital_circulante = activo_corriente - pasivo_corriente

    deuda_lp = serie_num_anual(df_feature_base, 'Deudas a largo plazo', anio)
    deuda_cp = serie_num_anual(df_feature_base, 'Deudas a corto plazo', anio)
    deuda_bancaria_lp = serie_num_anual(df_feature_base, 'Deudas con entidades de credito', anio, 0)
    deuda_bancaria_cp = serie_num_anual(df_feature_base, 'Deudas con entidades de credito', anio, 1)
    leasing_lp = serie_num_anual(df_feature_base, 'Acreedores por arrendamiento financiero', anio, 0)
    leasing_cp = serie_num_anual(df_feature_base, 'Acreedores por arrendamiento financiero', anio, 1)
    deuda_financiera = deuda_lp + deuda_cp + leasing_lp.fillna(0) + leasing_cp.fillna(0)
    deuda_bancaria = deuda_bancaria_lp.fillna(0) + deuda_bancaria_cp.fillna(0)

    ventas = serie_num_anual(df_feature_base, 'Importe neto de la cifra de negocios', anio)
    otros_ingresos = serie_num_anual(df_feature_base, 'Otros ingresos de explotacion', anio)
    ingresos_financieros = serie_num_anual(df_feature_base, 'Ingresos financieros', anio)
    ingresos_totales = ventas + otros_ingresos.fillna(0) + ingresos_financieros.fillna(0)

    deudores = serie_num_anual(df_feature_base, 'Deudores comerciales y otras cuentas a cobrar', anio)
    acreedores = serie_num_anual(df_feature_base, 'Acreedores comerciales y otras cuentas a pagar', anio)
    aprovisionamientos = serie_num_anual(df_feature_base, 'Aprovisionamientos', anio)
    gastos_personal = serie_num_anual(df_feature_base, 'Gastos de personal', anio)
    otros_gastos = serie_num_anual(df_feature_base, 'Otros gastos de explotacion', anio)
    gastos_operativos = aprovisionamientos.abs() + gastos_personal.abs() + otros_gastos.abs()

    inmovilizado_intangible = serie_num_anual(df_feature_base, 'Inmovilizado intangible', anio)
    inmovilizado_material = serie_num_anual(df_feature_base, 'Inmovilizado material', anio)
    inversiones_inmobiliarias = serie_num_anual(df_feature_base, 'Inversiones inmobiliarias', anio)
    activos_fijos_operativos = (
        inmovilizado_intangible.fillna(0)
        + inmovilizado_material.fillna(0)
        + inversiones_inmobiliarias.fillna(0)
    )

    ebit = serie_num_anual(df_feature_base, 'Resultado de explotacion', anio)
    amortizacion = serie_num_anual(df_feature_base, 'Amortizacion del inmovilizado', anio)
    ebitda = ebit - amortizacion  # amortizacion esta registrada como gasto negativo en la base.
    gastos_financieros = serie_num_anual(df_feature_base, 'Gastos financieros', anio)
    resultado = serie_num_anual(df_feature_base, 'Resultado del ejercicio', anio, 0)
    empleados = serie_num_anual(df_feature_base, 'Numero empleados', anio)

    if anio == YEARS[0]:
        variacion_capital_circulante = pd.Series(np.nan, index=df_feature_base.index)
        capex_proxy = pd.Series(np.nan, index=df_feature_base.index)
        free_cash_flow = pd.Series(np.nan, index=df_feature_base.index)
    else:
        anio_anterior = anio - 1
        activo_corriente_anterior = serie_num_anual(df_feature_base, 'Activo corriente', anio_anterior)
        pasivo_corriente_anterior = serie_num_anual(df_feature_base, 'Pasivo corriente', anio_anterior)
        capital_circulante_anterior = activo_corriente_anterior - pasivo_corriente_anterior
        activos_fijos_operativos_anterior = (
            serie_num_anual(df_feature_base, 'Inmovilizado intangible', anio_anterior).fillna(0)
            + serie_num_anual(df_feature_base, 'Inmovilizado material', anio_anterior).fillna(0)
            + serie_num_anual(df_feature_base, 'Inversiones inmobiliarias', anio_anterior).fillna(0)
        )
        variacion_capital_circulante = capital_circulante - capital_circulante_anterior
        capex_proxy = (activos_fijos_operativos - activos_fijos_operativos_anterior) + amortizacion.abs()
        free_cash_flow = ebit + amortizacion.abs() - variacion_capital_circulante - capex_proxy

    prefijo = f'{anio}_'
    features_anuales[prefijo + 'log_total_assets'] = log1p_no_negativo(total_activo)
    features_anuales[prefijo + 'log_sales'] = log1p_no_negativo(ventas)
    features_anuales[prefijo + 'log_employees'] = log1p_no_negativo(empleados)
    features_anuales[prefijo + 'equity'] = patrimonio_neto
    features_anuales[prefijo + 'current_ratio'] = dividir_seguro(activo_corriente, pasivo_corriente)
    features_anuales[prefijo + 'quick_ratio'] = dividir_seguro(activo_corriente - existencias, pasivo_corriente)
    features_anuales[prefijo + 'cash_ratio'] = dividir_seguro(efectivo, pasivo_corriente)
    features_anuales[prefijo + 'working_capital_to_assets'] = dividir_seguro(activo_corriente - pasivo_corriente, total_activo)
    features_anuales[prefijo + 'cash_to_assets'] = dividir_seguro(efectivo, total_activo)
    features_anuales[prefijo + 'inventory_to_sales'] = dividir_seguro(existencias, ventas)
    features_anuales[prefijo + 'receivables_to_sales'] = dividir_seguro(deudores, ventas)
    features_anuales[prefijo + 'payables_to_sales'] = dividir_seguro(acreedores, ventas)
    features_anuales[prefijo + 'payables_to_assets'] = dividir_seguro(acreedores, total_activo)
    features_anuales[prefijo + 'dso'] = dividir_seguro(deudores, ventas) * 365
    features_anuales[prefijo + 'dio'] = dividir_seguro(existencias, aprovisionamientos.abs()) * 365
    features_anuales[prefijo + 'dpo'] = dividir_seguro(acreedores, aprovisionamientos.abs()) * 365
    features_anuales[prefijo + 'cash_conversion_cycle'] = features_anuales[prefijo + 'dso'] + features_anuales[prefijo + 'dio'] - features_anuales[prefijo + 'dpo']
    features_anuales[prefijo + 'total_liabilities_to_assets'] = dividir_seguro(pasivo_total, total_activo)
    features_anuales[prefijo + 'financial_debt_to_assets'] = dividir_seguro(deuda_financiera, total_activo)
    features_anuales[prefijo + 'financial_debt_to_ebitda'] = dividir_seguro(deuda_financiera, ebitda)
    features_anuales[prefijo + 'net_debt_to_ebitda'] = dividir_seguro(deuda_financiera - efectivo, ebitda)
    features_anuales[prefijo + 'debt_to_equity'] = dividir_seguro(pasivo_total, patrimonio_neto)
    features_anuales[prefijo + 'financial_debt_to_equity'] = dividir_seguro(deuda_financiera, patrimonio_neto)
    features_anuales[prefijo + 'short_term_debt_share'] = dividir_seguro(deuda_cp, deuda_lp + deuda_cp)
    features_anuales[prefijo + 'bank_debt_share'] = dividir_seguro(deuda_bancaria, deuda_lp + deuda_cp)
    features_anuales[prefijo + 'ebit_margin'] = dividir_seguro(ebit, ventas)
    features_anuales[prefijo + 'ebitda_margin'] = dividir_seguro(ebitda, ventas)
    features_anuales[prefijo + 'net_margin'] = dividir_seguro(resultado, ventas)
    features_anuales[prefijo + 'roa'] = dividir_seguro(resultado, total_activo)
    features_anuales[prefijo + 'roe'] = dividir_seguro(resultado, patrimonio_neto)
    features_anuales[prefijo + 'interest_coverage_ebit'] = dividir_seguro(ebit, gastos_financieros.abs())
    features_anuales[prefijo + 'interest_coverage_ebitda'] = dividir_seguro(ebitda, gastos_financieros.abs())
    features_anuales[prefijo + 'financial_expense_to_sales'] = dividir_seguro(gastos_financieros.abs(), ventas)
    features_anuales[prefijo + 'financial_expense_to_revenue'] = dividir_seguro(gastos_financieros.abs(), ingresos_totales)
    features_anuales[prefijo + 'operating_cost_to_sales'] = dividir_seguro(gastos_operativos, ventas)
    features_anuales[prefijo + 'personnel_cost_to_sales'] = dividir_seguro(gastos_personal.abs(), ventas)
    features_anuales[prefijo + 'free_cash_flow'] = free_cash_flow
    features_anuales[prefijo + 'free_cash_flow_to_sales'] = dividir_seguro(free_cash_flow, ventas)
    features_anuales[prefijo + 'free_cash_flow_to_assets'] = dividir_seguro(free_cash_flow, total_activo)

    features_anuales[prefijo + 'loss_flag'] = (resultado < 0).astype(int)
    features_anuales[prefijo + 'negative_ebit_flag'] = (ebit < 0).astype(int)
    features_anuales[prefijo + 'negative_equity_flag'] = (patrimonio_neto < 0).astype(int)
    features_anuales[prefijo + 'liabilities_greater_than_assets_flag'] = (pasivo_total > total_activo).astype(int)
    features_anuales[prefijo + 'zero_sales_flag'] = (ventas.abs() <= 1e-9).astype(int)
    features_anuales[prefijo + 'zero_equity_flag'] = (patrimonio_neto.abs() <= 1e-9).astype(int)
    features_anuales[prefijo + 'zero_financial_expense_flag'] = (gastos_financieros.abs() <= 1e-9).astype(int)
    features_anuales[prefijo + 'high_leverage_flag'] = (features_anuales[prefijo + 'total_liabilities_to_assets'] > 0.8).astype(int)

    features_anuales[prefijo + 'has_financial_investments_lp'] = (serie_num_anual(df_feature_base, 'Inversiones financieras a largo plazo', anio).fillna(0).abs() > 0).astype(int)
    features_anuales[prefijo + 'has_financial_investments_cp'] = (serie_num_anual(df_feature_base, 'Inversiones financieras a corto plazo', anio).fillna(0).abs() > 0).astype(int)
    features_anuales[prefijo + 'has_deferred_tax_assets'] = (serie_num_anual(df_feature_base, 'Activos por impuesto diferido', anio).fillna(0).abs() > 0).astype(int)
    features_anuales[prefijo + 'has_deferred_tax_liabilities'] = (serie_num_anual(df_feature_base, 'Pasivos por impuesto diferido', anio).fillna(0).abs() > 0).astype(int)
    features_anuales[prefijo + 'has_provisions_lp'] = (serie_num_anual(df_feature_base, 'Provisiones a largo plazo', anio).fillna(0).abs() > 0).astype(int)
    features_anuales[prefijo + 'has_provisions_cp'] = (serie_num_anual(df_feature_base, 'Provisiones a corto plazo', anio).fillna(0).abs() > 0).astype(int)
    features_anuales[prefijo + 'has_exchange_differences'] = (serie_num_anual(df_feature_base, 'Diferencias de cambio', anio).fillna(0).abs() > 0).astype(int)
    features_anuales[prefijo + 'has_special_debt'] = (
        serie_num_anual(df_feature_base, 'Deuda con caracteristicas especiales', anio, 0).fillna(0).abs()
        + serie_num_anual(df_feature_base, 'Deuda con caracteristicas especiales', anio, 1).fillna(0).abs()
        > 0
    ).astype(int)
    features_anuales[prefijo + 'has_group_debt'] = (
        serie_num_anual(df_feature_base, 'Deudas con empresas del grupo', anio, 0).fillna(0).abs()
        + serie_num_anual(df_feature_base, 'Deudas con empresas del grupo', anio, 1).fillna(0).abs()
        > 0
    ).astype(int)
    features_anuales[prefijo + 'has_group_receivables'] = (
        serie_num_anual(df_feature_base, 'Inversiones en empresas del grupo', anio, 0).fillna(0).abs()
        + serie_num_anual(df_feature_base, 'Inversiones en empresas del grupo', anio, 1).fillna(0).abs()
        > 0
    ).astype(int)

features_anuales = pd.DataFrame(features_anuales, index=df_feature_base.index).replace([np.inf, -np.inf], np.nan)

features_agregadas = {}
metricas = sorted({col.split('_', 1)[1] for col in features_anuales.columns if col[:4].isdigit()})
for metrica in metricas:
    cols = [f'{anio}_{metrica}' for anio in YEARS]
    serie_anual = features_anuales[cols]
    features_agregadas[f'{metrica}_mean_4y'] = serie_anual.mean(axis=1)
    features_agregadas[f'{metrica}_median_4y'] = serie_anual.median(axis=1)
    features_agregadas[f'{metrica}_min_4y'] = serie_anual.min(axis=1)
    features_agregadas[f'{metrica}_max_4y'] = serie_anual.max(axis=1)
    features_agregadas[f'{metrica}_std_4y'] = serie_anual.std(axis=1)
    features_agregadas[f'{metrica}_delta_2011_2008'] = serie_anual[f'2011_{metrica}'] - serie_anual[f'2008_{metrica}']
    features_agregadas[f'{metrica}_pct_change_2011_2008'] = dividir_seguro(
        serie_anual[f'2011_{metrica}'] - serie_anual[f'2008_{metrica}'],
        serie_anual[f'2008_{metrica}'].abs()
    )
    features_agregadas[f'{metrica}_slope_2008_2011'] = (serie_anual[f'2011_{metrica}'] - serie_anual[f'2008_{metrica}']) / 3

for flag in [
    'loss_flag', 'negative_ebit_flag', 'negative_equity_flag',
    'liabilities_greater_than_assets_flag', 'zero_sales_flag',
    'zero_equity_flag', 'zero_financial_expense_flag', 'high_leverage_flag'
]:
    features_agregadas[f'{flag}_count_4y'] = features_anuales[[f'{anio}_{flag}' for anio in YEARS]].sum(axis=1)

features_agregadas = pd.DataFrame(features_agregadas, index=df_feature_base.index).replace([np.inf, -np.inf], np.nan)
features_financieras = pd.concat([features_anuales, features_agregadas], axis=1)

def contar_descensos(metrica):
    anual = features_financieras[[f'{anio}_{metrica}' for anio in YEARS]]
    return (anual.diff(axis=1).iloc[:, 1:] < 0).sum(axis=1)


def contar_incrementos(metrica):
    anual = features_financieras[[f'{anio}_{metrica}' for anio in YEARS]]
    return (anual.diff(axis=1).iloc[:, 1:] > 0).sum(axis=1)

features_tendencia = pd.DataFrame({
    'sales_declined_2_or_more_years': (contar_descensos('log_sales') >= 2).astype(int),
    'assets_declined_2_or_more_years': (contar_descensos('log_total_assets') >= 2).astype(int),
    'cash_declined_2_or_more_years': (contar_descensos('cash_to_assets') >= 2).astype(int),
    'equity_declined_2_or_more_years': (contar_descensos('equity') >= 2).astype(int),
    'debt_increased_2_or_more_years': (contar_incrementos('financial_debt_to_assets') >= 2).astype(int),
    'consecutive_loss_years': racha_maxima_unos(features_financieras[[f'{anio}_loss_flag' for anio in YEARS]])
}, index=df_feature_base.index)

auditoria_2011 = serie_cat_anual(df_feature_base, 'Calificacion auditor', 2011).str.lower().fillna('__missing__')
auditoria_2008 = serie_cat_anual(df_feature_base, 'Calificacion auditor', 2008).str.lower().fillna('__missing__')
severidad_auditoria = {'aprobado': 0, 'salvedades': 1, 'desfavor.': 2, 'denegado': 3, '__missing__': np.nan}
sev_2011 = auditoria_2011.map(severidad_auditoria)
sev_2008 = auditoria_2008.map(severidad_auditoria)
features_auditoria = pd.DataFrame({
    'audit_worse_than_approved_flag_2011': (~auditoria_2011.eq('aprobado')).astype(int),
    'audit_denied_flag_2011': auditoria_2011.eq('denegado').astype(int),
    'audit_changed_flag_2008_2011': (~auditoria_2008.eq(auditoria_2011)).astype(int),
    'audit_worsened_flag_2008_2011': (sev_2011 > sev_2008).astype(int),
    'audit_severity_2011': sev_2011,
}, index=df_feature_base.index)
features_auditoria = pd.concat([
    features_auditoria,
    pd.get_dummies(auditoria_2011, prefix='audit_opinion_2011', dtype=int)
], axis=1)

y_modelo = df_feature_base['Y']
folds_target_encoding = crear_folds_estratificados(y_modelo)
cnae_2011 = serie_cat_anual(df_feature_base, 'Codigo primario CNAE 2009', 2011).fillna('__missing__')
provincia_2011 = serie_cat_anual(df_feature_base, 'Provincia', 2011).fillna('__missing__')
cnae_2digitos = cnae_2011.astype('string').str.slice(0, 2).fillna('__missing__')
provincia_cnae = provincia_2011.astype('string') + '__' + cnae_2digitos.astype('string')
features_target_encoding = pd.DataFrame({
    'cnae_default_rate_oof': target_encoding_oof(cnae_2011, y_modelo, folds_target_encoding),
    'province_default_rate_oof': target_encoding_oof(provincia_2011, y_modelo, folds_target_encoding),
    'cnae_2digit_default_rate_oof': target_encoding_oof(cnae_2digitos, y_modelo, folds_target_encoding),
    'province_cnae_default_rate_oof': target_encoding_oof(provincia_cnae, y_modelo, folds_target_encoding),
}, index=df_feature_base.index)

features_sector_relativo = pd.DataFrame(index=df_feature_base.index)
for nueva_col, col_base in {
    'current_ratio_vs_cnae_median': '2011_current_ratio',
    'debt_assets_vs_cnae_median': '2011_total_liabilities_to_assets',
    'roa_vs_cnae_median': '2011_roa',
    'sales_growth_vs_cnae_median': 'log_sales_pct_change_2011_2008',
    'cash_assets_vs_cnae_median': '2011_cash_to_assets',
}.items():
    mediana_sector = features_financieras.groupby(cnae_2011)[col_base].transform('median')
    features_sector_relativo[nueva_col] = features_financieras[col_base] - mediana_sector.fillna(features_financieras[col_base].median())

features_invalidos = pd.DataFrame(index=df_feature_base.index)
columnas_posibles_ratios = [
    col for col in features_financieras.columns
    if any(token in col for token in ['ratio', 'to_', 'coverage', 'margin', 'dso', 'dio', 'dpo'])
]
for col in columnas_posibles_ratios:
    if features_financieras[col].isna().any():
        features_invalidos[f'{col}_was_invalid_flag'] = features_financieras[col].isna().astype(int)

df_modelo_impagos = pd.concat([
    df_feature_base[['ID', 'Y']].reset_index(drop=True),
    features_financieras.reset_index(drop=True),
    features_tendencia.reset_index(drop=True),
    features_auditoria.reset_index(drop=True),
    features_target_encoding.reset_index(drop=True),
    features_sector_relativo.reset_index(drop=True),
    features_invalidos.reset_index(drop=True),
], axis=1)

df_modelo_impagos = df_modelo_impagos.replace([np.inf, -np.inf], np.nan)
df_modelo_impagos.to_csv('fusionado_features_impagos.csv', index=False, encoding='utf-8-sig')

print(f'df_modelo_impagos -> filas: {df_modelo_impagos.shape[0]}, columnas: {df_modelo_impagos.shape[1]}')
print(f'Tasa de impago: {df_modelo_impagos["Y"].mean():.2%}')
try:
    display(df_modelo_impagos.head())
except NameError:
    print(df_modelo_impagos.head())


## Logit PCS

### Pipeline actual

```text
140 features (train+test)
        |
Imputacion + escalado
        |
PCA no supervisado
73,8% varianza - 30 comp.
        |
Logit umbral fijo 0,5
Sin ajuste por desbalanceo
        |
Resultados
ROC-AUC 0,72 - Recall 0,07
3 impagos detectados / 42
```

### Pipeline propuesto

```text
140 features (train+test)
        |
Imputacion + escalado
        |
Seleccion supervisada
LASSO / RFE sobre target
        |
Modelo + desbalanceo
class_weight / SMOTE / umbral
        |
Calibracion de umbral
        |
Objetivo
ROC-AUC > 0,75 - Recall > 0,4
F1 equilibrado
```

El bloque siguiente implementa la parte propuesta: seleccion supervisada con regularizacion L1, modelo logistico con `class_weight='balanced'` y calibracion del umbral sobre train.


In [ ]:
import statsmodels.api as sm
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, confusion_matrix,
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, brier_score_loss
)
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

df_logit = df_modelo_impagos.copy()

# Features documentadas en README: valor reciente 2011 y mediana temporal de 4 años.
features_base_readme = [
    'log_total_assets', 'log_sales', 'log_employees',
    'current_ratio', 'quick_ratio', 'cash_ratio', 'working_capital_to_assets', 'cash_to_assets',
    'total_liabilities_to_assets', 'financial_debt_to_assets', 'financial_debt_to_ebitda',
    'net_debt_to_ebitda', 'debt_to_equity', 'financial_debt_to_equity',
    'short_term_debt_share', 'bank_debt_share',
    'interest_coverage_ebit', 'interest_coverage_ebitda',
    'financial_expense_to_sales', 'financial_expense_to_revenue',
    'ebit_margin', 'ebitda_margin', 'net_margin', 'roa', 'roe',
    'dso', 'dio', 'dpo', 'cash_conversion_cycle',
    'inventory_to_sales', 'receivables_to_sales', 'payables_to_sales', 'payables_to_assets',
    'operating_cost_to_sales', 'personnel_cost_to_sales',
    'loss_flag', 'negative_ebit_flag', 'negative_equity_flag',
    'liabilities_greater_than_assets_flag', 'high_leverage_flag',
    'zero_sales_flag', 'zero_equity_flag', 'zero_financial_expense_flag',
    'has_financial_investments_lp', 'has_financial_investments_cp',
    'has_deferred_tax_assets', 'has_deferred_tax_liabilities',
    'has_provisions_lp', 'has_provisions_cp', 'has_exchange_differences',
    'has_special_debt', 'has_group_debt', 'has_group_receivables',
]

columnas_logit_readme = []
for feature in features_base_readme:
    columnas_logit_readme.extend([f'2011_{feature}', f'{feature}_median_4y'])
    if feature.endswith('_flag'):
        columnas_logit_readme.append(f'{feature}_count_4y')

columnas_logit_readme.extend([
    'sales_declined_2_or_more_years', 'assets_declined_2_or_more_years',
    'cash_declined_2_or_more_years', 'equity_declined_2_or_more_years',
    'debt_increased_2_or_more_years', 'consecutive_loss_years',
    'audit_worse_than_approved_flag_2011', 'audit_denied_flag_2011',
    'audit_changed_flag_2008_2011', 'audit_worsened_flag_2008_2011',
    'audit_severity_2011',
    'cnae_default_rate_oof', 'province_default_rate_oof',
    'cnae_2digit_default_rate_oof', 'province_cnae_default_rate_oof',
    'current_ratio_vs_cnae_median', 'debt_assets_vs_cnae_median',
    'roa_vs_cnae_median', 'sales_growth_vs_cnae_median', 'cash_assets_vs_cnae_median',
])

columnas_logit_readme.extend([col for col in df_logit.columns if col.startswith('audit_opinion_2011_')])
columnas_logit_readme.extend([
    f'{col}_was_invalid_flag'
    for col in columnas_logit_readme
    if f'{col}_was_invalid_flag' in df_logit.columns
])

columnas_esperadas_logit_readme = list(dict.fromkeys(columnas_logit_readme))
faltantes_logit_readme = [col for col in columnas_esperadas_logit_readme if col not in df_logit.columns]
if faltantes_logit_readme:
    print(f'Features del README no encontradas y omitidas: {faltantes_logit_readme}')
columnas_logit_readme = [col for col in columnas_esperadas_logit_readme if col in df_logit.columns]

X_raw = df_logit[columnas_logit_readme].select_dtypes(include=[np.number])
datos_modelo = pd.concat([df_logit['Y'], X_raw], axis=1).dropna(subset=['Y'])
Y = datos_modelo['Y'].astype(int)
X_raw = datos_modelo.drop(columns=['Y'])

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    Y,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=Y
)

columnas_validas = X_train_raw.columns[X_train_raw.notna().any()].tolist()
X_train_raw = X_train_raw[columnas_validas]
X_test_raw = X_test_raw[columnas_validas]

medianas_train = X_train_raw.median(numeric_only=True)
X_train = X_train_raw.fillna(medianas_train)
X_test = X_test_raw.fillna(medianas_train)

columnas_no_constantes = X_train.columns[X_train.nunique(dropna=False) > 1].tolist()
X_train = X_train[columnas_no_constantes]
X_test = X_test[columnas_no_constantes]

medias_train = X_train.mean()
std_train = X_train.std(ddof=0).replace(0, 1)
X_train = (X_train - medias_train) / std_train
X_test = (X_test - medias_train) / std_train

selector_lasso = SelectFromModel(
    LogisticRegression(
        penalty='l1',
        solver='saga',
        class_weight='balanced',
        max_iter=5000,
        random_state=RANDOM_SEED,
    ),
    max_features=min(30, X_train.shape[1]),
    threshold='median'
)
selector_lasso.fit(X_train, y_train)
columnas_seleccionadas = X_train.columns[selector_lasso.get_support()].tolist()

if not columnas_seleccionadas:
    modelo_importancia = LogisticRegression(
        penalty='l2',
        solver='liblinear',
        class_weight='balanced',
        max_iter=2000,
        random_state=RANDOM_SEED,
    ).fit(X_train, y_train)
    importancia = pd.Series(
        np.abs(modelo_importancia.coef_[0]),
        index=X_train.columns
    ).sort_values(ascending=False)
    columnas_seleccionadas = importancia.head(min(30, X_train.shape[1])).index.tolist()

X_train_sel = X_train[columnas_seleccionadas]
X_test_sel = X_test[columnas_seleccionadas]

modelo_logit = LogisticRegression(
    penalty='l2',
    solver='liblinear',
    class_weight='balanced',
    max_iter=2000,
    random_state=RANDOM_SEED,
)
modelo_logit.fit(X_train_sel, y_train)

y_prob_train = modelo_logit.predict_proba(X_train_sel)[:, 1]
y_prob_test = modelo_logit.predict_proba(X_test_sel)[:, 1]

umbrales = np.linspace(0.05, 0.95, 181)
metricas_umbral_train = []
for umbral in umbrales:
    y_pred_train_umbral = (y_prob_train >= umbral).astype(int)
    metricas_umbral_train.append({
        'umbral': umbral,
        'precision': precision_score(y_train, y_pred_train_umbral, zero_division=0),
        'recall': recall_score(y_train, y_pred_train_umbral, zero_division=0),
        'f1': f1_score(y_train, y_pred_train_umbral, zero_division=0),
    })
metricas_umbral_train = pd.DataFrame(metricas_umbral_train)

candidatos_umbral = metricas_umbral_train[metricas_umbral_train['recall'] >= 0.40]
if candidatos_umbral.empty:
    fila_umbral = metricas_umbral_train.sort_values(['f1', 'recall'], ascending=False).iloc[0]
else:
    fila_umbral = candidatos_umbral.sort_values(['f1', 'precision'], ascending=False).iloc[0]

umbral_logit = float(fila_umbral['umbral'])
y_pred_test = (y_prob_test >= umbral_logit).astype(int)

metricas_logit_test = pd.Series({
    'accuracy': accuracy_score(y_test, y_pred_test),
    'balanced_accuracy': balanced_accuracy_score(y_test, y_pred_test),
    'precision': precision_score(y_test, y_pred_test, zero_division=0),
    'recall': recall_score(y_test, y_pred_test, zero_division=0),
    'f1': f1_score(y_test, y_pred_test, zero_division=0),
    'roc_auc': roc_auc_score(y_test, y_prob_test),
    'average_precision': average_precision_score(y_test, y_prob_test),
    'brier_score': brier_score_loss(y_test, y_prob_test),
    'threshold': umbral_logit,
})

matriz_confusion_logit_test = pd.DataFrame(
    confusion_matrix(y_test, y_pred_test),
    index=['real_0', 'real_1'],
    columns=['pred_0', 'pred_1']
)

predicciones_logit_test = pd.DataFrame({
    'Y_real': y_test,
    'prob_impago': y_prob_test,
    'Y_pred_calibrado': y_pred_test,
}, index=y_test.index).sort_values('prob_impago', ascending=False)

coeficientes_logit = pd.Series(
    modelo_logit.coef_[0],
    index=columnas_seleccionadas,
    name='coeficiente'
).sort_values(key=np.abs, ascending=False)

print(f'Observaciones train: {len(y_train)}')
print(f'Observaciones test: {len(y_test)}')
print(f'Tasa de impago train: {y_train.mean():.2%}')
print(f'Tasa de impago test: {y_test.mean():.2%}')
print(f'Features originales del README usadas antes de seleccion: {X_train.shape[1]}')
print(f'Features retenidas por seleccion supervisada: {len(columnas_seleccionadas)}')
print(f'Umbral calibrado: {umbral_logit:.3f}')
display(metricas_logit_test.to_frame('test'))
display(matriz_confusion_logit_test)
display(coeficientes_logit.head(20).to_frame())
display(predicciones_logit_test.head(10))


In [ ]:
from scipy.stats import norm
from sklearn.ensemble import RandomForestClassifier
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.model_selection import StratifiedKFold, cross_val_score

rng_rf = np.random.default_rng(RANDOM_SEED)

espacio_random_forest = {
    'n_estimators': [200, 300, 500, 700, 900],
    'max_depth': [None, 3, 5, 8, 12, 16],
    'min_samples_split': [2, 5, 10, 20, 40],
    'min_samples_leaf': [1, 2, 5, 10, 20],
    'max_features': ['sqrt', 'log2', 0.4, 0.6, 0.8, 1.0],
    'bootstrap': [True, False],
    'class_weight': ['balanced', 'balanced_subsample'],
}

def muestrear_parametros_rf(n_muestras, rng):
    claves = list(espacio_random_forest)
    parametros = []
    vistos = set()
    while len(parametros) < n_muestras:
        candidato = {}
        for clave in claves:
            valores = espacio_random_forest[clave]
            valor = valores[int(rng.integers(len(valores)))]
            candidato[clave] = valor.item() if hasattr(valor, 'item') else valor
        firma = tuple(candidato[clave] for clave in claves)
        if firma not in vistos:
            vistos.add(firma)
            parametros.append(candidato)
    return parametros

def codificar_parametros_rf(parametros):
    filas = []
    for p in parametros:
        filas.append([
            p['n_estimators'],
            0 if p['max_depth'] is None else p['max_depth'],
            p['min_samples_split'],
            p['min_samples_leaf'],
            {'sqrt': 0.2, 'log2': 0.3}.get(p['max_features'], p['max_features']),
            int(p['bootstrap']),
            int(p['class_weight'] == 'balanced_subsample'),
        ])
    return np.asarray(filas, dtype=float)

cv_random_forest = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cache_scores_rf = {}

def evaluar_parametros_rf(parametros):
    firma = tuple(sorted(parametros.items()))
    if firma not in cache_scores_rf:
        modelo = RandomForestClassifier(
            **parametros,
            random_state=RANDOM_SEED,
            n_jobs=1,
        )
        cache_scores_rf[firma] = cross_val_score(
            modelo,
            X_train_sel,
            y_train,
            scoring='average_precision',
            cv=cv_random_forest,
            n_jobs=-1,
        ).mean()
    return cache_scores_rf[firma]

n_inicial_random_forest = 12
n_iter_bayes_random_forest = 28
n_candidatos_random_forest = 300

historial_random_forest = []
for parametros in muestrear_parametros_rf(n_inicial_random_forest, rng_rf):
    historial_random_forest.append({**parametros, 'cv_average_precision': evaluar_parametros_rf(parametros)})

for _ in range(n_iter_bayes_random_forest):
    parametros_observados = [
        {clave: fila[clave] for clave in espacio_random_forest}
        for fila in historial_random_forest
    ]
    X_bo = codificar_parametros_rf(parametros_observados)
    y_bo = np.array([fila['cv_average_precision'] for fila in historial_random_forest])

    gp = GaussianProcessRegressor(
        kernel=ConstantKernel(1.0) * Matern(nu=2.5) + WhiteKernel(noise_level=1e-5),
        normalize_y=True,
        random_state=RANDOM_SEED,
    )
    gp.fit(X_bo, y_bo)

    candidatos = muestrear_parametros_rf(n_candidatos_random_forest, rng_rf)
    X_candidatos = codificar_parametros_rf(candidatos)
    media, std = gp.predict(X_candidatos, return_std=True)
    mejora = media - y_bo.max()
    z = np.divide(mejora, std, out=np.zeros_like(mejora), where=std > 0)
    expected_improvement = mejora * norm.cdf(z) + std * norm.pdf(z)
    mejor_candidato = candidatos[int(np.argmax(expected_improvement))]
    historial_random_forest.append({
        **mejor_candidato,
        'cv_average_precision': evaluar_parametros_rf(mejor_candidato),
    })

resultados_bayes_random_forest = pd.DataFrame(historial_random_forest).sort_values(
    'cv_average_precision',
    ascending=False
)
mejores_parametros_random_forest = {
    clave: resultados_bayes_random_forest.iloc[0][clave]
    for clave in espacio_random_forest
}
mejores_parametros_random_forest.update({
    'n_estimators': int(mejores_parametros_random_forest['n_estimators']),
    'max_depth': None if pd.isna(mejores_parametros_random_forest['max_depth']) else int(mejores_parametros_random_forest['max_depth']),
    'min_samples_split': int(mejores_parametros_random_forest['min_samples_split']),
    'min_samples_leaf': int(mejores_parametros_random_forest['min_samples_leaf']),
    'bootstrap': bool(mejores_parametros_random_forest['bootstrap']),
})

modelo_random_forest = RandomForestClassifier(
    **mejores_parametros_random_forest,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
modelo_random_forest.fit(X_train_sel, y_train)

y_prob_rf_train = modelo_random_forest.predict_proba(X_train_sel)[:, 1]
y_prob_rf_test = modelo_random_forest.predict_proba(X_test_sel)[:, 1]

metricas_umbral_rf_train = []
for umbral in umbrales:
    y_pred_rf_train_umbral = (y_prob_rf_train >= umbral).astype(int)
    metricas_umbral_rf_train.append({
        'umbral': umbral,
        'precision': precision_score(y_train, y_pred_rf_train_umbral, zero_division=0),
        'recall': recall_score(y_train, y_pred_rf_train_umbral, zero_division=0),
        'f1': f1_score(y_train, y_pred_rf_train_umbral, zero_division=0),
    })
metricas_umbral_rf_train = pd.DataFrame(metricas_umbral_rf_train)

candidatos_umbral_rf = metricas_umbral_rf_train[metricas_umbral_rf_train['recall'] >= 0.40]
if candidatos_umbral_rf.empty:
    fila_umbral_rf = metricas_umbral_rf_train.sort_values(['f1', 'recall'], ascending=False).iloc[0]
else:
    fila_umbral_rf = candidatos_umbral_rf.sort_values(['f1', 'precision'], ascending=False).iloc[0]

umbral_random_forest = float(fila_umbral_rf['umbral'])
y_pred_rf_test = (y_prob_rf_test >= umbral_random_forest).astype(int)

metricas_random_forest_test = pd.Series({
    'accuracy': accuracy_score(y_test, y_pred_rf_test),
    'balanced_accuracy': balanced_accuracy_score(y_test, y_pred_rf_test),
    'precision': precision_score(y_test, y_pred_rf_test, zero_division=0),
    'recall': recall_score(y_test, y_pred_rf_test, zero_division=0),
    'f1': f1_score(y_test, y_pred_rf_test, zero_division=0),
    'roc_auc': roc_auc_score(y_test, y_prob_rf_test),
    'average_precision': average_precision_score(y_test, y_prob_rf_test),
    'brier_score': brier_score_loss(y_test, y_prob_rf_test),
    'threshold': umbral_random_forest,
})

matriz_confusion_random_forest_test = pd.DataFrame(
    confusion_matrix(y_test, y_pred_rf_test),
    index=['real_0', 'real_1'],
    columns=['pred_0', 'pred_1']
)

predicciones_random_forest_test = pd.DataFrame({
    'Y_real': y_test,
    'prob_impago': y_prob_rf_test,
    'Y_pred_calibrado': y_pred_rf_test,
}, index=y_test.index).sort_values('prob_impago', ascending=False)

importancias_random_forest = pd.Series(
    modelo_random_forest.feature_importances_,
    index=columnas_seleccionadas,
    name='importancia'
).sort_values(ascending=False)

print(f'Observaciones train: {len(y_train)}')
print(f'Observaciones test: {len(y_test)}')
print(f'Tasa de impago train: {y_train.mean():.2%}')
print(f'Tasa de impago test: {y_test.mean():.2%}')
print(f'Features usadas por Random Forest: {X_train_sel.shape[1]}')
print(f'Mejor average precision CV: {resultados_bayes_random_forest.iloc[0]["cv_average_precision"]:.4f}')
print(f'Umbral calibrado Random Forest: {umbral_random_forest:.3f}')
display(pd.Series(mejores_parametros_random_forest, name='valor').to_frame())
display(resultados_bayes_random_forest.head(10))
display(metricas_random_forest_test.to_frame('test'))
display(matriz_confusion_random_forest_test)
display(importancias_random_forest.head(20).to_frame())
display(predicciones_random_forest_test.head(10))


## Logit de liquidez


In [ ]:
columnas_liquidez = [
    'Y',
    'Current Ratio_mediana_4_anios_norm',
    'Quick Ratio_mediana_4_anios_norm',
    'Cash Ratio_mediana_4_anios_norm'
]

faltantes_liquidez = [col for col in columnas_liquidez if col not in df_4_fusionado.columns]
if faltantes_liquidez:
    raise KeyError(f'Faltan columnas para Logit de liquidez: {faltantes_liquidez}')

df_logit_liquidez = df_4_fusionado[columnas_liquidez].copy()
datos_logit_liquidez = df_logit_liquidez.dropna()

Y_liquidez = datos_logit_liquidez['Y']
X_liquidez = sm.add_constant(datos_logit_liquidez.drop(columns=['Y']), has_constant='add')

modelo_logit_liquidez = sm.Logit(Y_liquidez, X_liquidez).fit()
print(f'Observaciones usadas en Logit Liquidez: {len(datos_logit_liquidez)}')
display(df_logit_liquidez.head())
modelo_logit_liquidez.summary()
